<!-- nav -->
[← 8 · Sublanguages](08_00_Sublanguages.ipynb) · [Home](../../README.md) · [8.02 · Python Patterns →](08_02_Python_Patterns.ipynb)

# 8.01 · Python Types

Tutorial 8 showed each sublanguage as a dialect of its own. This page takes one, `Python Types`: what Python writes
where it means a type. A type annotation, a `TypeVar`'s bound, a type argument: the typing specification writes each as
a **type expression**, and a type expression is a Python expression, `list[int] | None`, which Python's own parser reads
as it reads any other. What Python 3.12 added beside them is not an expression: a **type parameter** of a generic
function, class or type alias, `T: int`, `*Ts` or `**P` in `def first[T: int](xs: list[T]) -> T`. `Python Types` holds
the three kinds of type parameter, and lists every kind of Python3's beside them, so that its holder holds any type
expression too ([CAP-276: Python's type expressions and match patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-276.md)).

This page builds type expressions and type parameters, from a name up, in the builders' fluent calls, prints each back
as Python writes it, validates them, finds where Python3's terms hold one, simplifies what can be simplified, and reads the
editions that have its kinds. Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed
and defined; `variable` writes a name, as tutorials 5 to 8 do. Where Python's text is an expression, Python also reads it,
with `Python3.Text.FromText`, and checks that it gives the same term (tutorial 3).

In [1]:
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from mbse.Expressions.Python import Python3, Types as PythonTypes
from toolkit import dialects

store = dialects()
S = store.Expressions
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))
TYPES = PythonTypes.Expressions.DIALECT


def refused(simplify, expression):
    """What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
    none (`NotEvaluated`, a `ValueError`)."""
    try:
        result = simplify(expression)
        return result.evaluate() if hasattr(result, "evaluate") else result
    except ValueError as error:
        return str(error)

<a id="T08.01.1"></a>
## 08.01.1 · A type expression is an expression

`int`, `list[int]` and `list[int] | None` are three type expressions, each a Python expression: a name, Python3's
`variable`; a subscript, Python3's `subscript`; and a union written with `|`, Python3's `binop`, whose right operand is
the singleton `None`, Python3's `constant` (tutorial 3). `Python Types`' holder, `S.Python.Types.Expression`, holds each
as its root. Its `Term` lists every branch of Python3's, so each root is a term of `Python Types`' union whose kind is
Python3's: `Terms.dialect_of` names the union that holds it, and the dialect's `owner` the dialect whose kind it is.
`Python Types` prints through Python3's printer (tutorial 8, 08.5), and Python reads each text into the same term:

In [2]:
none = lambda d: d.constant(lambda k: k.singleton("None").text("None"))  # the singleton None, a Python3 constant
listed = lambda d: d.subscript(lambda s: s.value(variable("list")).slice(variable("int")))  # list[int]
with store.transaction():
    types = {
        "int": S.Python.Types.Expression(lambda e: e.root(variable("int"))),
        "list[int]": S.Python.Types.Expression(lambda e: e.root(listed)),
        "list[int] | None": S.Python.Types.Expression(lambda e: e.root(
            lambda u: u.binop(lambda b: b.left(listed).op("|").right(none)))),
    }
    read = {source: S.Python.Types.Expression(lambda e, source=source: e.root(Python3.Text.FromText(source)))
            for source in types}  # Python reads them
for source, expression in types.items():
    term = Terms.term(expression)
    print(f"{PythonTypes.Text.ToText(expression):17}", f"{Terms.kind_of(term)[0]:10}", Terms.dialect_of(term),
          "| owner:", TYPES.owner(term).name())
    assert PythonTypes.Text.ToText(expression) == source and Terms.same(read[source], expression)
assert PythonTypes.Text.ToText is Python3.Text.ToText  # one printer for the family

int               variable   Expressions.Python.Types.Term | owner: Python3
list[int]         subscript  Expressions.Python.Types.Term | owner: Python3
list[int] | None  binop      Expressions.Python.Types.Term | owner: Python3


<a id="T08.01.2"></a>
## 08.01.2 · A type parameter

A type parameter declares a name a generic definition takes. Python writes three kinds, each `Python Types`' own, named
and shaped as mbse-programs has them (the Language Reference, 8.11, "Type parameter lists"):

```text
T                       a typevar: its name
T: int                  its bound, a type expression
T: (int, str)           its constraints, a tuple
T = str                 its default (Python 3.13)
*Ts                     a typevartuple
**P = [int, str]        a paramspec, with its default
T: list[int] | None = int
```

Each is built from the smallest up, its `name` an attribute, its `bound` and its `default_value` subterms, each a type
expression of Python3's kinds (08.01.1). A form reads a type parameter as it reads any term (tutorial 5): its kind, its
name, and its bound and default as its arguments, `None` where it has none:

In [3]:
with store.transaction():
    parameters = [
        S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T")))),
        S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T").bound(variable("int"))))),
        S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T").bound(
            lambda b: b.tuple(lambda p: p.elts([variable("int"), variable("str")])))))),
        S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T").default_value(variable("str"))))),
        S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevartuple(lambda t: t.name("Ts")))),
        S.Python.Types.Expression(lambda e: e.root(lambda u: u.paramspec(lambda t: t.name("P").default_value(
            lambda d: d.list(lambda p: p.elements([  # a list display: the framework's list, of the framework's terms
                lambda t: t.python3(variable("int")), lambda t: t.python3(variable("str"))])))))),
        S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T")
            .bound(lambda b: b.binop(lambda o: o.left(listed).op("|").right(none))).default_value(variable("int"))))),
    ]
for parameter in parameters:
    form = Terms.form(Terms.term(parameter))
    print(f"{PythonTypes.Text.ToText(parameter):27}", f"{form.kind:13}", f"name: {form.attributes['name']:3}", "| subterms:",
          ", ".join("-" if a is None else PythonTypes.Text.ToText(a) for a in form.arguments))
assert [PythonTypes.Text.ToText(p) for p in parameters] == [
    "T", "T: int", "T: (int, str)", "T = str", "*Ts", "**P = [int, str]", "T: list[int] | None = int"]

T                           typevar       name: T   | subterms: -, -
T: int                      typevar       name: T   | subterms: int, -
T: (int, str)               typevar       name: T   | subterms: (int, str), -
T = str                     typevar       name: T   | subterms: -, str
*Ts                         typevartuple  name: Ts  | subterms: -
**P = [int, str]            paramspec     name: P   | subterms: [int, str]
T: list[int] | None = int   typevar       name: T   | subterms: list[int] | None, int


A type parameter's text is no expression, so `FromText` reads only its bound and its default, which are; built around
them, the parameter is the one 08.01.2 built:

In [4]:
with store.transaction():
    read = S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T")
        .bound(Python3.Text.FromText("list[int] | None")).default_value(Python3.Text.FromText("int")))))
assert Terms.same(read, parameters[-1])
try:  # Python's own parser reads `T: int` as a statement, an annotated name, not as an expression
    Python3.Text.FromText("T: int")
    raise AssertionError("read")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

ValueError: the source must end with an expression


<a id="T08.01.3"></a>
## 08.01.3 · Validated, and what it binds

A type parameter declares its name for the statement that lists it, a generic `def`, `class` or `type`, which
mbse-programs models: within the term, it binds nothing. So the names its bound and its default read are references its
scope must give, its own name among them, as in `T: list[T]`; `Symbolics.free` gives them, and validation reports each
one not bound. Python3's builtins that read values, `int` and `str` among them, are always bound (tutorial 4); `list` is
not one, so the scope gives it. Validation checks each kind's own rules too, with its own message: a type parameter's
name must be an identifier, so `1T` is kept as written, printed, and refused
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [5]:
with store.transaction():
    recursive = S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T").bound(
        lambda b: b.subscript(lambda s: s.value(variable("list")).slice(variable("T")))))))
    misnamed = S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("1T"))))
for expression in (parameters[-1], recursive, misnamed):
    print(f"{PythonTypes.Text.ToText(expression):27}", "free:", ", ".join(sorted(Symbolics.free(expression, TYPES))) or "none",
          "|", "; ".join(TYPES.validate(expression)) or "valid")
assert TYPES.validate(parameters[-1], {"list"}) == TYPES.validate(recursive, {"list", "T"}) == []  # its scope gives them
assert TYPES.validate(misnamed) == ["a typevar's name must be an identifier, got '1T'"]

T: list[int] | None = int   free: list | bound: left: value: variable 'list' is not bound
T: list[T]                  free: T, list | bound: value: variable 'list' is not bound; bound: slice: variable 'T' is not bound
1T                          free: none | a typevar's name must be an identifier, got '1T'


<a id="T08.01.4"></a>
## 08.01.4 · Where Python3 holds one

Python writes a type expression where it annotates: a parameter's annotation, a function's return type, a variable's.
Those are statements and definitions, mbse-programs', but a parameter is a helper Python3 shares with them, `arg`
(tutorial 6), whose `annotation` is a subterm of `Python Types`' `Term`. So Python3 reads `Python Types`' terms, as
each dialect of the family reads the others', `reads()`. The one expression with parameters, `lambda`, takes no
annotations in Python's grammar: built with one, it is held as written, and validation says what Python's parser would:

In [6]:
with store.transaction():
    annotated = S.Python.Python3.Expression(lambda e: e.root(lambda u: getattr(u, "lambda")(lambda l: l
        .args(lambda a: a.args([lambda g: g.arg("x").annotation(lambda t: t.binop(
            lambda b: b.left(variable("int")).op("|").right(none)))]))
        .body(variable("x")))))
annotation = Terms.form(Terms.term(annotated)).arguments[2]  # after Let's empty value and its body: x's annotation
print("Python3 reads", ", ".join(d.name() for d in Python3.Expressions.DIALECT.reads()))
print(f"{Python3.Text.ToText(annotation):10}", Terms.dialect_of(annotation), "| owner:",
      Python3.Expressions.DIALECT.owner(annotation).name())
print(*Python3.Expressions.DIALECT.validate(annotated))

Python3 reads Python Types, Python Patterns, Python RegExp
int | None Expressions.Python.Types.Term | owner: Python3
a lambda's parameters have no annotations


<a id="T08.01.5"></a>
## 08.01.5 · Simplified as Python3's, or left in the residual, named

A type expression is simplified as the Python3 expression it is, by Python3's rules, in a scope that gives its names:
`int if wide else float` chooses a type by a flag. A type parameter is not simplified: a statement declares it, and
Python3's rules fold none of `Python Types`' kinds: each stays in the residual, named by the dialect (tutorial 8, 08.3):

In [7]:
with store.transaction():
    chosen = S.Python.Types.Expression(lambda e: e.root(lambda u: u.ifexp(lambda i: i
        .test(variable("wide")).body(variable("int")).orelse(variable("float")))))
scope = {"wide": True, "int": "int64", "float": "float32"}
print(f"{PythonTypes.Text.ToText(chosen):24}", Python3.Simplifiers.OfAny(chosen, scope).evaluate())
for parameter in (parameters[0], parameters[4], parameters[5]):
    print(f"{PythonTypes.Text.ToText(parameter):24}", refused(Python3.Simplifiers.OfAny, parameter))

int if wide else float   int64
T                        Python Types typevar is not evaluated
*Ts                      Python Types typevartuple is not evaluated
**P = [int, str]         Python Types paramspec is not evaluated


<a id="T08.01.6"></a>
## 08.01.6 · Its editions as data

`Python Types` covers the versions of Python that Python3 covers, 3.0 to 3.15, each found by its published name
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)),
and records which of its own kinds each has: none before Python 3.12, which added type parameter lists, all three
since. The kinds it lists from Python3 are Python3's, recorded there (tutorial 2, 02.1):

In [8]:
dialect = store.Instances.Expressions.Python.Types
editions = Standards.standards(dialect)
print(len(editions), "editions:", ", ".join(list(editions)[:2]), "...", list(editions)[-1])
for version in ("Python 3.11", "Python 3.12", "Python 3.15"):
    print(f"{version:12}", ", ".join(kind.name.rsplit(".", 1)[1] for kind in Standards.kinds(Standards.find(dialect, version)))
          or "none of its own")

16 editions: Python 3.0, Python 3.1 ... Python 3.15
Python 3.11  none of its own
Python 3.12  TypeVar, ParamSpec, TypeVarTuple
Python 3.15  TypeVar, ParamSpec, TypeVarTuple


## What you learned

- **A type expression is a Python expression**, of Python3's kinds, held by `Python Types`' holder, whose `Term` lists
  every branch of Python3's; Python reads its text and prints it back.
- **A type parameter is `Python Types`' own**: `typevar`, `typevartuple` and `paramspec`, a name with a bound and a
  default, each a type expression.
- **It binds nothing within a term**: the statement that lists it binds its name, so the names its bound and default
  read are references, which validation reports where no scope gives them.
- **Python3 holds a type expression** where a parameter is annotated, and reads `Python Types`' terms there.
- **A type expression simplifies as Python3's; a type parameter stays in the residual**, named by its dialect.
- **Its editions are data**: Python 3.0 to 3.15, its own kinds since 3.12.

Next, `Python Patterns`: the patterns of Python's `match` statement.

<!-- nav -->
[← 8 · Sublanguages](08_00_Sublanguages.ipynb) · [Home](../../README.md) · [8.02 · Python Patterns →](08_02_Python_Patterns.ipynb)